# Блок 8 · Практика по текстам и RAG
## «Умный помощник по учебным документам» — решённый ноутбук

Ноутбук повторяет обязательную часть исходного `Block08_RAG_Text_TODO`
и использует те же модули, что консольное приложение `main.py`.

Что сделано по экзаменационным пунктам:

1. Обязательные шаги выполняются сверху вниз без ошибок.
2. Изменены четыре параметра: `CHUNK_SIZE`, `OVERLAP`, `SEARCH_METHOD`, `TOP_K`.
3. Добавлен пятый эксперимент `E` и объяснение по графикам.
4. Добавлен свой документ `STUDENT` и свой вопрос к нему.
5. Показан вопрос, на который система не находит подтверждённого ответа.
6. Результаты сохраняются в `reports/` (CSV, JSON, TXT, PNG, markdown).
7. В конце — объяснение различий TF-IDF, нейропоиска и RAG с LLM.

> Это поисковая часть RAG. Пока не подключена языковая модель, ответ
> собирается из точных цитат и не называется генерацией нейросетью.

## 1. Подключаем библиотеки и проект

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt

from src.config import (
    CHECK_QUESTIONS_PATH,
    CHUNK_SIZE,
    CHUNKS_CHART,
    CUSTOM_ANSWER_TXT,
    CUSTOM_QUESTION,
    DEMO_ANSWER_TXT,
    DEMO_QUESTION,
    DOCUMENTS_PATH,
    EXPERIMENTS_CHART,
    EXPERIMENTS_CSV,
    FINAL_REPORT_PATH,
    NO_ANSWER_TXT,
    OVERLAP,
    REPORTS_DIR,
    CHARTS_DIR,
    RETRIEVAL_CHECK_CSV,
    SEARCH_METHOD,
    SETTINGS_JSON,
    TOP_K,
    UNKNOWN_MIN_SIMILARITY,
    UNKNOWN_QUESTION,
)
from src.answerer import NO_ANSWER
from src.assistant import RagAssistant
from src.data_loader import DataLoader
from src.evaluator import SearchEvaluator
from src.experiments import ExperimentRunner
from src.report_builder import ReportBuilder
from src.visualizer import Visualizer

pd.set_option("display.max_colwidth", 80)
print("Project root:", PROJECT_ROOT)
print("Параметры:", CHUNK_SIZE, OVERLAP, SEARCH_METHOD, TOP_K)

## 2. Учебная база знаний

16 документов из ноутбука плюс собственный документ `STUDENT`.

In [ ]:
documents = DataLoader(DOCUMENTS_PATH).load_documents()
check_questions = DataLoader.load_check_questions(CHECK_QUESTIONS_PATH)

source_table = pd.DataFrame([
    {
        "источник": doc["id"],
        "название": doc["title"],
        "символов": len(doc["text"]),
    }
    for doc in documents
])
display(source_table)
print("Документов:", len(documents), "| контрольных вопросов:", len(check_questions))
assert len(documents) >= 17
assert any(doc["id"] == "STUDENT" for doc in documents)

## 3. Разбиение на чанки и TF-IDF индекс

Рабочие параметры изменены относительно исходного ноутбука:
`CHUNK_SIZE=230`, `OVERLAP=30`, `SEARCH_METHOD='char_wb'`, `TOP_K=3`.

In [ ]:
assistant = RagAssistant(
    documents,
    chunk_size=CHUNK_SIZE,
    overlap=OVERLAP,
    search_method=SEARCH_METHOD,
    top_k=TOP_K,
)
print("Чанков:", len(assistant.records))
print("Первые чанки:")
display(pd.DataFrame([
    {
        "chunk_id": record["chunk_id"],
        "doc_id": record["doc_id"],
        "символов": len(record["text"]),
        "начало": record["text"][:100] + "…",
    }
    for record in assistant.records[:8]
]))
assert len(assistant.records) >= len(documents)

## 4. Демонстрационный вопрос и источники

In [ ]:
demo_result = assistant.ask(DEMO_QUESTION)
print("ВАШ ВОПРОС:", DEMO_QUESTION)
display(pd.DataFrame([
    {
        "место": number,
        "документ": item["doc_id"],
        "название": item["title"],
        "сходство": round(float(item["score"]), 3),
        "фрагмент": item["text"][:150] + "…",
    }
    for number, item in enumerate(demo_result["found"], 1)
]))
print("\nОТВЕТ С ИСТОЧНИКАМИ (точные выдержки):")
print(demo_result["answer"])
assert demo_result["found"][0]["doc_id"] == "D04"
assert demo_result["answer"] != NO_ANSWER

## 5. Свой документ и свой вопрос

Документ `STUDENT` описывает правила консольного помощника. Вопрос:
«Какой флаг включает интерактивный режим консольного помощника?»

In [ ]:
custom_result = assistant.ask(CUSTOM_QUESTION)
print("СВОЙ ВОПРОС:", CUSTOM_QUESTION)
display(pd.DataFrame([
    {
        "место": number,
        "документ": item["doc_id"],
        "название": item["title"],
        "сходство": round(float(item["score"]), 3),
    }
    for number, item in enumerate(custom_result["found"], 1)
]))
print(custom_result["answer"])
assert custom_result["found"][0]["doc_id"] == "STUDENT"
assert "--ask" in custom_result["answer"]

## 6. Вопрос без подтверждённого ответа

Заведомо посторонний вопрос. Порог сходства поднят до `0.20`, чтобы
система не выдала случайный фрагмент за ответ.

In [ ]:
unknown_result = assistant.ask(
    UNKNOWN_QUESTION, min_similarity=UNKNOWN_MIN_SIMILARITY
)
print("ПОСТОРОННИЙ ВОПРОС:", UNKNOWN_QUESTION)
print("Лучший фрагмент:")
display(pd.DataFrame([
    {
        "документ": item["doc_id"],
        "название": item["title"],
        "сходство": round(float(item["score"]), 3),
    }
    for item in unknown_result["found"]
]))
print("ОТВЕТ:", unknown_result["answer"])
assert unknown_result["answer"] == NO_ANSWER

## 7. Проверка поиска: Top-1 и Hit@K

Контрольные вопросы имеют заранее известный документ-ответ.

In [ ]:
detail_df, summary = SearchEvaluator.check(
    assistant.retriever, check_questions, top_k=TOP_K
)
display(detail_df)
print("Top-1:", round(summary["top1_accuracy"], 3),
      "| Hit@K:", round(summary["hit_at_k"], 3))
assert summary["top1_accuracy"] <= summary["hit_at_k"] + 1e-9

## 8. Пять экспериментов

A–D из ноутбука и новый опыт `E`: поиск по словам, `overlap=0`,
`top_k=5`.

In [ ]:
results_df, experiment_details = ExperimentRunner(documents, check_questions).run()
display(results_df)
assert results_df.shape[0] == 5
assert results_df["hit_at_k"].between(0, 1).all()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
labels = [name.split(":", 1)[0] for name in results_df["опыт"]]

axes[0].bar(labels, results_df["top1"], color="#4C72B0")
axes[0].set_title("Правильный документ на первом месте")
axes[0].set_ylabel("Доля правильных ответов")

axes[1].bar(labels, results_df["hit_at_k"], color="#55A868")
axes[1].set_title("Правильный документ среди TOP-K")
axes[1].set_ylabel("Доля правильных ответов")

for ax in axes:
    ax.set_ylim(0, 1.02)
    ax.grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

### Вывод по графикам

- `char_wb` (B) лучше словарного поиска (A) на русских словоформах.
- Короткие чанки (C) дают лучший Top-1: они ближе к одному предложению-ответу.
- Длинные чанки (D) сохраняют контекст, но размывают сходство.
- Новый E с `top_k=5` поднимает Hit@K, но Top-1 падает: больше источников
  добавляет шум. `overlap=0` уменьшает число чанков и может рвать мысль
  на границе.

## 9. Сохраняем результаты для защиты и GitHub

In [ ]:
for directory in (REPORTS_DIR, CHARTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

DataLoader.save_dataframe(results_df, EXPERIMENTS_CSV)
DataLoader.save_dataframe(detail_df, RETRIEVAL_CHECK_CSV)
DataLoader.save_text(
    "Вопрос: " + DEMO_QUESTION + "\n" + demo_result["answer"],
    DEMO_ANSWER_TXT,
)
DataLoader.save_text(
    "Вопрос: " + CUSTOM_QUESTION + "\n" + custom_result["answer"],
    CUSTOM_ANSWER_TXT,
)
DataLoader.save_text(
    "Вопрос: " + UNKNOWN_QUESTION + "\n" + unknown_result["answer"],
    NO_ANSWER_TXT,
)

settings = {
    "chunk_size": CHUNK_SIZE,
    "overlap": OVERLAP,
    "search_method": SEARCH_METHOD,
    "top_k": TOP_K,
    "question": DEMO_QUESTION,
    "top1_accuracy": summary["top1_accuracy"],
    "hit_at_k": summary["hit_at_k"],
    "unknown_min_similarity": UNKNOWN_MIN_SIMILARITY,
    "custom_question": CUSTOM_QUESTION,
    "custom_source": custom_result["found"][0]["doc_id"],
    "documents": len(documents),
    "chunks": len(assistant.records),
    "note": "TF-IDF: поиск и подтверждённые выдержки; LLM выключена",
}
DataLoader.save_json(settings, SETTINGS_JSON)

visualizer = Visualizer(CHARTS_DIR)
visualizer.save_experiment_metrics(
    results_df, EXPERIMENTS_CHART.name, "Сравнение настроек поиска"
)
visualizer.save_chunks_per_experiment(
    results_df, CHUNKS_CHART.name, "Количество чанков"
)

limitations = [
    "База знаний небольшая и учебная: 17 документов.",
    "TF-IDF не понимает синонимы без общих слов.",
    "Ответ собирается из цитат, а не генерируется LLM.",
    "Порог сходства нужно настраивать на своих вопросах.",
    "Семантический поиск и локальная LLM — факультатив.",
]
report_builder = ReportBuilder(FINAL_REPORT_PATH)
report_text = report_builder.build_report(
    documents=documents,
    settings=settings,
    results_df=results_df,
    baseline_summary=summary,
    custom_result=custom_result,
    unknown_result=unknown_result,
    limitations=limitations,
)
report_builder.save(report_text)
print("Сохранено в:", REPORTS_DIR)

## 10. Факультатив: нейросетевой поиск

Если установлен `sentence-transformers` и есть интернет, можно сравнить
эмбеддинги с TF-IDF. Без пакета ячейка безопасно сообщает о пропуске.

In [ ]:
try:
    from src.neural_retriever import NeuralRetriever

    if not NeuralRetriever.is_available():
        print("sentence-transformers не установлен — факультатив пропущен.")
    else:
        neural = NeuralRetriever().build(assistant.records)
        neural_found = neural.search(DEMO_QUESTION, top_k=TOP_K)
        display(pd.DataFrame([
            {
                "источник": item["doc_id"],
                "название": item["title"],
                "сходство": round(float(item["score"]), 3),
            }
            for item in neural_found
        ]))
except (ImportError, OSError) as exc:
    print("Факультатив пропущен:", str(exc)[:250])

## 11. Итог и объяснение для защиты

**Что делает помощник.** Делит документы на чанки, строит TF-IDF индекс,
находит ближайшие фрагменты и выдаёт точные предложения с номерами
источников. При слабом сходстве отвечает «не нашёл подтверждённого
ответа».

**TF-IDF.** Статистический поиск по словам или группам символов. Не
обучает глубокую нейросеть. Не понимает синонимы, если нет общих слов.

**Нейросетевой поиск.** Превращает текст в эмбеддинги предобученной
нейросетью и сравнивает их по смыслу. Может найти «секретный ключ» по
фразе «токены доступа», но требует ресурсов и тоже ошибается.

**Полноценный RAG с LLM.** После поиска фрагменты передаются языковой
модели, которая пишет новый ответ. Появляется генерация, но и риск
галлюцинаций: ответ обязательно проверяют по источникам.

**Ограничения проекта.** Маленькая учебная база, TF-IDF без семантики,
ответ — цитаты, а не генерация. Найденная цитата не заменяет проверку
прав доступа и содержания.